# Liquidity Ratios — LCR and NSFR

CRR Art. 412–428 (LCR) and CRR Art. 428a–428ax (NSFR).
<small>

| Ratio | Formula | Minimum |
|---|---|---|
| LCR  | HQLA / Net Cash Outflows (30-day stress) | ≥ 100% |
| NSFR | Available Stable Funding / Required Stable Funding (1-year) | ≥ 100% |

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from banking_risk.liquidity import (
    # LCR
    HQLA_Asset, HQLA_Level, HQLA_HAIRCUTS,
    Cash_Outflow, Cash_Inflow,
    Outflow_Type, Inflow_Type, OUTFLOW_RATES,
    LCR_Result, SA_LCR_Calculator,
    # NSFR
    ASF_Item, ASF_Category,
    RSF_Item, RSF_Category,
    NSFR_Result, SA_NSFR_Calculator,
)
from banking_risk.utils.reporting import Dark_Style

## 2. Liquidity Coverage Ratio (LCR)

### 2a. HQLA portfolio

The liquidity buffer held against a 30-day stress outflow. Level caps apply:
- Level 2B ≤ 15% of HQLA
- Level 2 (2A + 2B) ≤ 40% of HQLA

The calculator solves these caps in closed form — no iteration needed.

In [ ]:
hqla_assets = [
    HQLA_Asset("Cash + CB reserves",   HQLA_Level.L1,            200_000_000),
    HQLA_Asset("ECB-eligible Govts",   HQLA_Level.L1,            300_000_000),
    HQLA_Asset("Covered bonds AA",     HQLA_Level.L2A,           120_000_000),
    HQLA_Asset("IG Corporate bonds",   HQLA_Level.L2B_CORPORATE,  60_000_000),
    HQLA_Asset("Qualifying RMBS",      HQLA_Level.L2B_RMBS,       30_000_000),
]

# Show pre-haircut vs adjusted values
rows = []
for a in hqla_assets:
    rows.append({
        "asset"        : a.name,
        "level"        : a.level,
        "market_value" : a.market_value / 1e6,
        "haircut_%"    : HQLA_HAIRCUTS[a.level] * 100,
        "adj_value_M"  : a.adjusted_value / 1e6,
    })
pd.DataFrame(rows).set_index("asset").round(2)

### 2b. Cash outflows and inflows

Prescribed rates from Commission Delegated Regulation (EU) 2015/61.

In [ ]:
outflows = [
    Cash_Outflow("Retail stable deposits",      1_000_000_000, Outflow_Type.RETAIL_STABLE),
    Cash_Outflow("Retail less stable deposits",   300_000_000, Outflow_Type.RETAIL_LESS_STABLE),
    Cash_Outflow("Operational deposits",          200_000_000, Outflow_Type.OPERATIONAL_DEPOSIT),
    Cash_Outflow("Non-fin corporate deposits",    150_000_000, Outflow_Type.NON_FINANCIAL_CORP),
    Cash_Outflow("Financial institution deposits",  80_000_000, Outflow_Type.FINANCIAL_INST),
    Cash_Outflow("Committed credit facilities",   100_000_000, Outflow_Type.COMMITTED_CREDIT),
    Cash_Outflow("Committed liquidity facilities",  50_000_000, Outflow_Type.COMMITTED_LIQUIDITY),
]

inflows = [
    Cash_Inflow("Retail loan repayments",    40_000_000, Inflow_Type.RETAIL),
    Cash_Inflow("Wholesale receivables",     60_000_000, Inflow_Type.WHOLESALE),
]

# Show outflow amounts
out_df = pd.DataFrame([
    {"item": o.name, "balance_M": o.balance/1e6, "rate_%": o.outflow_rate*100,
     "outflow_M": o.stressed_amount/1e6}
    for o in outflows
]).set_index("item")
print(f"Total gross outflows: EUR {out_df['outflow_M'].sum():,.1f}M")
out_df.round(1)

### 2c. LCR result

In [ ]:
lcr_result = SA_LCR_Calculator().compute(hqla_assets, outflows, inflows)

print(f"HQLA (after caps)  : EUR {lcr_result.hqla/1e6:>8.1f}M")
print(f"  Level 1          : EUR {lcr_result.hqla_l1/1e6:>8.1f}M")
print(f"  Level 2A         : EUR {lcr_result.hqla_l2a/1e6:>8.1f}M")
print(f"  Level 2B         : EUR {lcr_result.hqla_l2b/1e6:>8.1f}M")
print(f"Gross outflows     : EUR {lcr_result.gross_outflows/1e6:>8.1f}M")
print(f"Gross inflows      : EUR {lcr_result.gross_inflows/1e6:>8.1f}M")
print(f"Net outflows       : EUR {lcr_result.net_outflows/1e6:>8.1f}M")
print(f"LCR                : {lcr_result.lcr:.1%}")
print(f"Status             : {'PASS ✓' if lcr_result.passes else 'FAIL ✗'}  (min 100%)")

In [ ]:
Dark_Style().apply()
p = Dark_Style().palette

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: HQLA composition
levels = ["L1", "L2A", "L2B"]
vals_m = [lcr_result.hqla_l1/1e6, lcr_result.hqla_l2a/1e6, lcr_result.hqla_l2b/1e6]
bar_colors = [p["cyan"], p["green"], p["amber"]]
bars = ax1.bar(levels, vals_m, color=bar_colors, alpha=0.85, width=0.5)
ax1.axhline(lcr_result.net_outflows/1e6, color=p["red"], lw=1.5, ls="--",
            label=f"Net outflows {lcr_result.net_outflows/1e6:.0f}M")
for bar, v in zip(bars, vals_m):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 2, f"{v:.0f}M",
             ha="center", fontsize=9, color=p["text_body"])
ax1.set_ylabel("EUR millions")
ax1.set_title("HQLA by level (after caps)", color=p["text_title"])
ax1.legend(fontsize=8)

# Panel 2: Outflow breakdown
labels = [o.name.replace(" deposits", "").replace(" facilities", "") for o in outflows]
amounts = [o.stressed_amount/1e6 for o in outflows]
ax2.barh(labels, amounts, color=p["red"], alpha=0.75)
ax2.set_xlabel("EUR millions")
ax2.set_title("30-day stressed outflows", color=p["text_title"])
ax2.grid(axis="x", alpha=0.3)

fig.suptitle(f"LCR = {lcr_result.lcr:.1%}  |  {'PASS' if lcr_result.passes else 'FAIL'}",
             color=p["green"] if lcr_result.passes else p["red"], fontweight="bold")
fig.tight_layout()
plt.show()

## 3. Net Stable Funding Ratio (NSFR)

The NSFR matches the stability of funding sources (ASF) to the liquidity
needs of assets (RSF) over a one-year stress horizon.

Key principle: long-term illiquid assets must be funded by stable, long-term liabilities.

In [ ]:
asf_items = [
    ASF_Item("Tier 1 capital",              200_000_000, category=ASF_Category.TIER1_CAPITAL),
    ASF_Item("Retail stable deposits >6M",  800_000_000, category=ASF_Category.RETAIL_STABLE_GT6M),
    ASF_Item("Retail stable deposits <6M",  200_000_000, category=ASF_Category.RETAIL_STABLE_LT6M),
    ASF_Item("Less stable deposits >6M",    150_000_000, category=ASF_Category.RETAIL_LESS_STABLE_GT6M),
    ASF_Item("Corporate deposits >6M",      100_000_000, category=ASF_Category.CORP_NON_FIN_GT6M),
    ASF_Item("Corporate deposits <6M",       80_000_000, category=ASF_Category.CORP_NON_FIN_LT6M),
    ASF_Item("Financial deposits >6M",       50_000_000, category=ASF_Category.FIN_INSTITUTION_GT6M),
    ASF_Item("Financial deposits <6M",       40_000_000, category=ASF_Category.FIN_INSTITUTION_LT6M),
]

rsf_items = [
    RSF_Item("Central bank reserves",       150_000_000, category=RSF_Category.CENTRAL_BANK_RESERVES),
    RSF_Item("Level 1 HQLA",                200_000_000, category=RSF_Category.HQLA_L1_UNENCUMBERED),
    RSF_Item("Level 2A HQLA",               100_000_000, category=RSF_Category.HQLA_L2A_UNENCUMBERED),
    RSF_Item("Retail loans <1Y",             80_000_000, category=RSF_Category.LOANS_RETAIL_LT1Y),
    RSF_Item("Residential mortgages >1Y",   400_000_000, category=RSF_Category.LOANS_RETAIL_MORTGAGE_GT1Y),
    RSF_Item("Corporate loans >1Y",         350_000_000, category=RSF_Category.LOANS_CORPORATE_GT1Y),
    RSF_Item("Other assets",                 80_000_000, category=RSF_Category.OTHER_ASSETS),
]

nsfr_result = SA_NSFR_Calculator().compute(asf_items, rsf_items)

print(f"ASF  : EUR {nsfr_result.available_stable_funding/1e6:>8.1f}M")
print(f"RSF  : EUR {nsfr_result.required_stable_funding/1e6:>8.1f}M")
print(f"NSFR : {nsfr_result.nsfr:.1%}")
print(f"Status: {'PASS ✓' if nsfr_result.passes else 'FAIL ✗'}  (min 100%)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: ASF waterfall
asf_d = nsfr_result.asf_detail.copy()
asf_d["asf_M"] = asf_d["asf"] / 1e6
labels = [n[:25] for n in asf_d["name"]]
ax1.barh(labels, asf_d["asf_M"], color=p["cyan"], alpha=0.85)
ax1.set_xlabel("ASF contribution (EUR M)")
ax1.set_title("Available Stable Funding", color=p["text_title"])
ax1.grid(axis="x", alpha=0.3)

# Panel 2: RSF waterfall
rsf_d = nsfr_result.rsf_detail.copy()
rsf_d["rsf_M"] = rsf_d["rsf"] / 1e6
labels_r = [n[:25] for n in rsf_d["name"]]
ax2.barh(labels_r, rsf_d["rsf_M"], color=p["amber"], alpha=0.85)
ax2.set_xlabel("RSF requirement (EUR M)")
ax2.set_title("Required Stable Funding", color=p["text_title"])
ax2.grid(axis="x", alpha=0.3)

fig.suptitle(f"NSFR = {nsfr_result.nsfr:.1%}  |  ASF {nsfr_result.available_stable_funding/1e6:.0f}M / RSF {nsfr_result.required_stable_funding/1e6:.0f}M",
             color=p["green"] if nsfr_result.passes else p["red"], fontweight="bold")
fig.tight_layout()
plt.show()

### 3a. NSFR sensitivity — funding mix impact

In [ ]:
# How NSFR changes as wholesale short-term funding replaces retail deposits
base_retail = 800_000_000
wholesale_shares = np.linspace(0, 0.60, 50)
nsfrs = []

for ws in wholesale_shares:
    retail_m  = base_retail * (1 - ws)
    wholesale = base_retail * ws
    asf_test = [
        ASF_Item("Tier1",    200_000_000, category=ASF_Category.TIER1_CAPITAL),
        ASF_Item("Retail",   retail_m,    category=ASF_Category.RETAIL_STABLE_GT6M),
        ASF_Item("Wholesale", wholesale,  category=ASF_Category.FIN_INSTITUTION_LT6M),
        ASF_Item("Other",    200_000_000, category=ASF_Category.CORP_NON_FIN_GT6M),
    ]
    r = SA_NSFR_Calculator().compute(asf_test, rsf_items)
    nsfrs.append(r.nsfr * 100)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(wholesale_shares * 100, nsfrs, color=p["cyan"], lw=2)
ax.axhline(100, color=p["red"], lw=1.2, ls="--", label="Regulatory minimum 100%")
ax.fill_between(wholesale_shares * 100, nsfrs, 100,
                where=[n < 100 for n in nsfrs], color=p["red"], alpha=0.2)
ax.set_xlabel("Short-term wholesale funding share (%)")
ax.set_ylabel("NSFR (%)")
ax.set_title("NSFR degradation as wholesale replaces retail funding",
             color=p["text_title"])
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()